In [1]:
import pandas as pd

df_temp1 = pd.read_csv('../temp_25_26.csv')
df_temp2 = pd.read_csv('../temp_24_25.csv')
df_temp3 = pd.read_csv('../temp_23_24.csv')
df_temp4 = pd.read_csv('../temp_22_23.csv')
df_temp5 = pd.read_csv('../temp_21_22.csv')
df_temp6 = pd.read_csv('../temp_20_21.csv') 

df_master = pd.concat([df_temp1, df_temp2, df_temp3, df_temp4, df_temp5, df_temp6], ignore_index=True)

print(f"Total de partidos (filas) y variables (columnas): {df_master.shape}")
df_master.head()

Total de partidos (filas) y variables (columnas): (2189, 161)


,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,BFA,1XBH,1XBD,1XBA,BFCH,BFCD,BFCA,1XBCH,1XBCD,1XBCA
0,I1,23/08/2025,17:30,Genoa,Lecce,0,0,D,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,I1,23/08/2025,17:30,Sassuolo,Napoli,0,2,A,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,I1,23/08/2025,19:45,Milan,Cremonese,1,2,A,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,I1,23/08/2025,19:45,Roma,Bologna,1,0,H,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,I1,24/08/2025,17:30,Cagliari,Fiorentina,1,1,D,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
columnas_clave = [
    'Date', 'HomeTeam', 'AwayTeam', 
    'FTHG', 'FTAG', 'HTHG', 'HTAG', 
    'HS', 'AS', 'HST', 'AST', 
    'HC', 'AC', 'HF', 'AF', 
    'HY', 'AY', 'HR', 'AR'
]

df_def = df_master[columnas_clave].copy()

print(f"Tabla: {df_def.shape}")
df_def.head()

Tabla: (2189, 19)


,Date,HomeTeam,AwayTeam,FTHG,FTAG,HTHG,HTAG,HS,AS,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR
0,23/08/2025,Genoa,Lecce,0,0,0,0,5,7,2,0,3,7,16,13,2,3,0,0
1,23/08/2025,Sassuolo,Napoli,0,2,0,1,7,13,2,4,1,2,17,17,3,1,1,0
2,23/08/2025,Milan,Cremonese,1,2,1,1,24,4,6,3,9,2,10,14,1,4,0,0
3,23/08/2025,Roma,Bologna,1,0,0,0,14,10,4,2,2,4,15,13,1,1,0,0
4,24/08/2025,Cagliari,Fiorentina,1,1,0,0,14,4,6,1,3,2,18,14,4,2,0,0


In [3]:
nulos_por_columna = df_def.isnull().sum()

columnas_con_nulos = nulos_por_columna[nulos_por_columna > 0]

print("DATOS FALTANTES")
if columnas_con_nulos.empty:
    print("No hay datos faltantes")
else:
    print("Hay datos faltantes en: ")
    print(columnas_con_nulos)
    
    filas_con_nulos = df_def.isnull().any(axis=1).sum()
    porcentaje = (filas_con_nulos / len(df_def)) * 100
    print(f"\nTotal de partidos con datos incompletos: {filas_con_nulos} ({porcentaje:.2f}%)")

DATOS FALTANTES
No hay datos faltantes


In [4]:
df_def['BTTS'] = ((df_def['FTHG'] > 0) & (df_def['FTAG'] > 0)).astype(int)
conteo = df_def['BTTS'].value_counts()
porcentaje = df_def['BTTS'].mean() * 100
print(f"Partidos donde NO anotaron ambos (0): {conteo[0]}")
print(f"Partidos donde SÍ anotaron ambos (1): {conteo[1]}")
print(f"Porcentaje histórico de 'Ambos Anotan': {porcentaje:.2f}%")

df_def[['HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'BTTS']].head()

Partidos donde NO anotaron ambos (0): 1022
Partidos donde SÍ anotaron ambos (1): 1167
Porcentaje histórico de 'Ambos Anotan': 53.31%


,HomeTeam,AwayTeam,FTHG,FTAG,BTTS
0,Genoa,Lecce,0,0,0
1,Sassuolo,Napoli,0,2,0
2,Milan,Cremonese,1,2,1
3,Roma,Bologna,1,0,0
4,Cagliari,Fiorentina,1,1,1


In [5]:
df_def['Date'] = pd.to_datetime(df_def['Date'], format='%d/%m/%Y')
df_def = df_def.sort_values(by='Date')
#Se calcula la racha de goles a favor y en contra de del equipo local
df_def['Racha_Local_Goles_Favor'] = df_def.groupby('HomeTeam')['FTHG'].transform(lambda x: x.rolling(window=5, closed='left').mean())
df_def['Racha_Local_Goles_Contra'] = df_def.groupby('HomeTeam')['FTAG'].transform(lambda x: x.rolling(window=5, closed='left').mean())

df_def['Racha_Visita_Goles_Favor'] = df_def.groupby('AwayTeam')['FTAG'].transform(lambda x: x.rolling(window=5, closed='left').mean())
df_def['Racha_Visita_Goles_Contra'] = df_def.groupby('AwayTeam')['FTHG'].transform(lambda x: x.rolling(window=5, closed='left').mean())

arsenal_visita = df_def[df_def['AwayTeam'] == 'Milan'][['Date', 'AwayTeam', 'FTAG', 'Racha_Visita_Goles_Favor']]
arsenal_visita.head(8)

,Date,AwayTeam,FTAG,Racha_Visita_Goles_Favor
1823,2020-09-27,Milan,2,NaN
1838,2020-10-17,Milan,2,NaN
1860,2020-11-01,Milan,2,NaN
1887,2020-11-22,Milan,3,NaN
1905,2020-12-06,Milan,2,NaN
1921,2020-12-16,Milan,2,2.2
1934,2020-12-20,Milan,2,2.2
1955,2021-01-03,Milan,2,2.2


In [6]:
df_def = df_def.dropna().reset_index(drop=True)

In [7]:
df_def['H2H_BTTS_Promedio'] = df_def.groupby(['HomeTeam', 'AwayTeam'])['BTTS'].transform(lambda x: x.shift(1).expanding().mean())

promedio_liga = df_def['BTTS'].mean()

df_def['H2H_BTTS_Promedio'] = df_def['H2H_BTTS_Promedio'].fillna(promedio_liga)

clasico = df_def[(df_def['HomeTeam'] == 'Milan') & (df_def['AwayTeam'] == 'Roma')][['Date', 'HomeTeam', 'AwayTeam', 'BTTS', 'H2H_BTTS_Promedio']]
clasico.head()

,Date,HomeTeam,AwayTeam,BTTS,H2H_BTTS_Promedio
442,2022-01-06,Milan,Roma,1,0.528736
769,2023-01-08,Milan,Roma,1,1.000000
1169,2024-01-14,Milan,Roma,1,1.000000
1514,2024-12-29,Milan,Roma,1,1.000000
1806,2025-11-02,Milan,Roma,0,1.000000


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Calculamos las nuevas rachas ofensivas (Últimos 5 partidos)
df_def['Racha_Local_TirosPuerta'] = df_def.groupby('HomeTeam')['HST'].transform(lambda x: x.rolling(window=5, closed='left').mean())
df_def['Racha_Visita_TirosPuerta'] = df_def.groupby('AwayTeam')['AST'].transform(lambda x: x.rolling(window=5, closed='left').mean())

# Córners (Tiros de esquina)
df_def['Racha_Local_Corners'] = df_def.groupby('HomeTeam')['HC'].transform(lambda x: x.rolling(window=5, closed='left').mean())
df_def['Racha_Visita_Corners'] = df_def.groupby('AwayTeam')['AC'].transform(lambda x: x.rolling(window=5, closed='left').mean())

# 2. Limpiamos los nuevos NaN que se generan por pedir 5 partidos más de historial
df_def = df_def.dropna().reset_index(drop=True)

# 3. ACTUALIZAMOS EL MODELO
columnas_features_v2 = [
    'Racha_Local_Goles_Favor', 'Racha_Local_Goles_Contra',
    'Racha_Visita_Goles_Favor', 'Racha_Visita_Goles_Contra',
    'H2H_BTTS_Promedio',
    'Racha_Local_TirosPuerta', 'Racha_Visita_TirosPuerta',
    'Racha_Local_Corners', 'Racha_Visita_Corners'
]

X = df_def[columnas_features_v2]
y = df_def['BTTS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo_rf_v2 = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf_v2.fit(X_train, y_train)

predicciones_v2 = modelo_rf_v2.predict(X_test)
precision_v2 = accuracy_score(y_test, predicciones_v2)

print(f"PRECISIÓN DEL MODELO 2.0: {precision_v2 * 100:.2f}%\n")

importancias_v2 = modelo_rf_v2.feature_importances_
print("¿Qué revisó más el modelo?")
for feature, imp in zip(columnas_features_v2, importancias_v2):
    print(f"{feature}: {imp * 100:.1f}%")

PRECISIÓN DEL MODELO 2.0: 50.83%

¿Qué revisó más el modelo?
Racha_Local_Goles_Favor: 10.0%
Racha_Local_Goles_Contra: 10.5%
Racha_Visita_Goles_Favor: 9.7%
Racha_Visita_Goles_Contra: 10.7%
H2H_BTTS_Promedio: 7.3%
Racha_Local_TirosPuerta: 12.9%
Racha_Visita_TirosPuerta: 12.4%
Racha_Local_Corners: 13.6%
Racha_Visita_Corners: 12.9%


In [9]:
import xgboost as xgb
from sklearn.metrics import accuracy_score

modelo_xgb = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
 
modelo_xgb.fit(X_train, y_train)

predicciones_xgb = modelo_xgb.predict(X_test)
precision_xgb = accuracy_score(y_test, predicciones_xgb)

print(f"PRECISIÓN DEL MODELO XGBOOST: {precision_xgb * 100:.2f}%\n")

importancias_xgb = modelo_xgb.feature_importances_
print("¿Qué revisó más XGBoost para decidir?")
for feature, imp in zip(columnas_features_v2, importancias_xgb):
    print(f"{feature}: {imp * 100:.1f}%")

PRECISIÓN DEL MODELO XGBOOST: 50.28%

¿Qué revisó más XGBoost para decidir?
Racha_Local_Goles_Favor: 12.1%
Racha_Local_Goles_Contra: 10.6%
Racha_Visita_Goles_Favor: 10.5%
Racha_Visita_Goles_Contra: 11.8%
H2H_BTTS_Promedio: 11.5%
Racha_Local_TirosPuerta: 11.3%
Racha_Visita_TirosPuerta: 9.9%
Racha_Local_Corners: 11.7%
Racha_Visita_Corners: 10.6%


In [10]:
df_master['Date'] = pd.to_datetime(df_master['Date'], dayfirst=True, errors='coerce')

def predecir_partido(local, visita, modelo, df_base):
    
    df_local = df_base[df_base['HomeTeam'] == local].sort_values('Date').tail(5)
    df_visita = df_base[df_base['AwayTeam'] == visita].sort_values('Date').tail(5)
    
    if len(df_local) < 5:
        print(f"Error: El equipo local '{local}' solo tiene {len(df_local)} partidos en tu base de datos.")
    if len(df_visita) < 5:
        print(f"Error: El equipo visitante '{visita}' solo tiene {len(df_visita)} partidos en tu base de datos.")
        
    racha_local_gf = df_local['FTHG'].mean()
    racha_local_gc = df_local['FTAG'].mean()
    racha_local_tp = df_local['HST'].mean()
    racha_local_cor = df_local['HC'].mean()
    
    racha_visita_gf = df_visita['FTAG'].mean()
    racha_visita_gc = df_visita['FTHG'].mean()
    racha_visita_tp = df_visita['AST'].mean()
    racha_visita_cor = df_visita['AC'].mean()
    
    h2h_historico = df_base[(df_base['HomeTeam'] == local) & (df_base['AwayTeam'] == visita)]
    if len(h2h_historico) > 0:
        btts_h2h = ((h2h_historico['FTHG'] > 0) & (h2h_historico['FTAG'] > 0)).mean()
    else:
        btts_h2h = 0.56 
        
    datos_hoy = pd.DataFrame([[
        racha_local_gf, racha_local_gc,
        racha_visita_gf, racha_visita_gc,
        btts_h2h,
        racha_local_tp, racha_visita_tp,
        racha_local_cor, racha_visita_cor
    ]], columns=columnas_features_v2) 
    
    probabilidades = modelo.predict_proba(datos_hoy)[0] 
    prob_no = probabilidades[0] * 100
    prob_si = probabilidades[1] * 100
    
    print(f"PROBABILIDAD DE QUE SÍ ANOTEN AMBOS: {prob_si:.2f}%")
    print(f"PROBABILIDAD DE QUE NO ANOTEN AMBOS: {prob_no:.2f}%")
    
    if prob_si > prob_no:
        print("PREDICCIÓN FINAL DEL MODELO: SÍ (BTTS = 1)")
    else:
        print("PREDICCIÓN FINAL DEL MODELO: NO (BTTS = 0)")


In [11]:
import requests
import pandas as pd

match_name_mapping = {
"Juventus FC": "Juventus",
    "AC Milan": "Milan",
    "FC Internazionale Milano": "Inter",
    "SSC Napoli": "Napoli",
    "AS Roma": "Roma",
    "SS Lazio": "Lazio",
    "ACF Fiorentina": "Fiorentina",
    "Atalanta BC": "Atalanta",
    "Bologna FC 1909": "Bologna",
    "Torino FC": "Torino",
    "Genoa CFC": "Genoa",
    "Hellas Verona FC": "Verona",
    "Empoli FC": "Empoli",
    "US Lecce": "Lecce",
    "Udinese Calcio": "Udinese",
    "Cagliari Calcio": "Cagliari",
    "AC Monza": "Monza",
    "Parma Calcio 1913": "Parma",
    "Venezia FC": "Venezia",
    "Como 1907": "Como",
    # Equipos de temporadas anteriores recientes por si están en tu historial:
    "US Sassuolo Calcio": "Sassuolo",
    "Frosinone Calcio": "Frosinone",
    "US Salernitana 1919": "Salernitana",
    "Spezia Calcio": "Spezia",
    "UC Sampdoria": "Sampdoria",
    "US Cremonese": "Cremonese",
    "AC Pisa 1909": "Pisa"

}

API_KEY = "65104bce29144179b5b85d53cf565eaa" 
URL_PL_MATCHES = "https://api.football-data.org/v4/competitions/SA/matches"

headers = {"X-Auth-Token": API_KEY}
params = {"status": "SCHEDULED", "limit": 10}

print("Obteniendo los próximos partidos de la API...")
respuesta = requests.get(URL_PL_MATCHES, headers=headers, params=params)

if respuesta.status_code == 200:
    lista_partidos = respuesta.json()["matches"][:10]
    
    print("\n" + "="*50)
    print("PREDICCIONES DE LA JORNADA (¿AMBOS ANOTAN?)")
    print("="*50)
    
    for partido in lista_partidos:
        fecha_utc = partido["utcDate"][:10]
        nombre_api_local = partido["homeTeam"]["name"]
        nombre_api_visitante = partido["awayTeam"]["name"]
        

        equipo_modelo_local = match_name_mapping.get(nombre_api_local, nombre_api_local)
        equipo_modelo_visitante = match_name_mapping.get(nombre_api_visitante, nombre_api_visitante)
        
        print(f"\nFecha: {fecha_utc}")
        print(f"Partido: {equipo_modelo_local} vs {equipo_modelo_visitante}")
        print("-" * 30)
        
        try:

            predecir_partido(equipo_modelo_local, equipo_modelo_visitante, modelo_xgb, df_master)
        except Exception as e:
            print(f"No se pudo predecir este partido. Detalles: {e}")
            
else:
    print(f"Error al conectar con la API. Código: {respuesta.status_code}")

Obteniendo los próximos partidos de la API...

PREDICCIONES DE LA JORNADA (¿AMBOS ANOTAN?)

Fecha: 2026-03-20
Partido: Cagliari vs Napoli
------------------------------
PROBABILIDAD DE QUE SÍ ANOTEN AMBOS: 32.59%
PROBABILIDAD DE QUE NO ANOTEN AMBOS: 67.41%
PREDICCIÓN FINAL DEL MODELO: NO (BTTS = 0)

Fecha: 2026-03-20
Partido: Genoa vs Udinese
------------------------------
PROBABILIDAD DE QUE SÍ ANOTEN AMBOS: 60.42%
PROBABILIDAD DE QUE NO ANOTEN AMBOS: 39.58%
PREDICCIÓN FINAL DEL MODELO: SÍ (BTTS = 1)

Fecha: 2026-03-21
Partido: Parma vs Cremonese
------------------------------
PROBABILIDAD DE QUE SÍ ANOTEN AMBOS: 69.12%
PROBABILIDAD DE QUE NO ANOTEN AMBOS: 30.88%
PREDICCIÓN FINAL DEL MODELO: SÍ (BTTS = 1)

Fecha: 2026-03-21
Partido: Milan vs Torino
------------------------------
PROBABILIDAD DE QUE SÍ ANOTEN AMBOS: 70.12%
PROBABILIDAD DE QUE NO ANOTEN AMBOS: 29.88%
PREDICCIÓN FINAL DEL MODELO: SÍ (BTTS = 1)

Fecha: 2026-03-21
Partido: Juventus vs Sassuolo
-----------------------------

In [12]:
import requests
import pandas as pd
from IPython.display import display

API_KEY = "65104bce29144179b5b85d53cf565eaa"
URL_PL_MATCHES = "https://api.football-data.org/v4/competitions/SA/matches"

headers = {"X-Auth-Token": API_KEY}
params = {"status": "SCHEDULED", "limit": 10}

respuesta = requests.get(URL_PL_MATCHES, headers=headers, params=params)

if respuesta.status_code == 200:
    lista_partidos = respuesta.json()["matches"][:10]

    resultados = []
    
    for partido in lista_partidos:
        fecha_utc = partido["utcDate"][:10]
        equipo_local = match_name_mapping.get(partido["homeTeam"]["name"], partido["homeTeam"]["name"])
        equipo_visita = match_name_mapping.get(partido["awayTeam"]["name"], partido["awayTeam"]["name"])
        
        df_local = df_master[df_master['HomeTeam'] == equipo_local].sort_values('Date').tail(5)
        df_visita = df_master[df_master['AwayTeam'] == equipo_visita].sort_values('Date').tail(5)
        
        if len(df_local) == 5 and len(df_visita) == 5:
            
            racha_local_gf = df_local['FTHG'].mean()
            racha_local_gc = df_local['FTAG'].mean()
            racha_local_tp = df_local['HST'].mean()
            racha_local_cor = df_local['HC'].mean()
            
            racha_visita_gf = df_visita['FTAG'].mean()
            racha_visita_gc = df_visita['FTHG'].mean()
            racha_visita_tp = df_visita['AST'].mean()
            racha_visita_cor = df_visita['AC'].mean()
            
            h2h = df_master[(df_master['HomeTeam'] == equipo_local) & (df_master['AwayTeam'] == equipo_visita)]
            btts_h2h = ((h2h['FTHG'] > 0) & (h2h['FTAG'] > 0)).mean() if len(h2h) > 0 else 0.56 
            
            datos_hoy = pd.DataFrame([[
                racha_local_gf, racha_local_gc, racha_visita_gf, racha_visita_gc, btts_h2h,
                racha_local_tp, racha_visita_tp, racha_local_cor, racha_visita_cor
            ]], columns=columnas_features_v2) 
            
            prob_si = modelo_xgb.predict_proba(datos_hoy)[0][1] * 100
            
            resultados.append({
                "Fecha": fecha_utc,
                "Local": equipo_local,
                "Visitante": equipo_visita,
                "% Prob. Ambos Anotan": f"{prob_si:.2f}%",
                "Predicción": "SÍ" if prob_si > 50 else "NO"
            })
            
        else:
            resultados.append({
                "Fecha": fecha_utc,
                "Local": equipo_local,
                "Visitante": equipo_visita,
                "% Prob. Ambos Anotan": "Faltan datos",
                "Predicción": "N/A"
            })

    df_tabla = pd.DataFrame(resultados)
    print("\nTABLA DE PREDICCIONES XGBOOST\n")
    df_tabla_estilo = df_tabla.style.set_properties(**{'text-align': 'center'}, subset=['% Prob. Ambos Anotan', 'Predicción'])
    display(df_tabla_estilo)

else:
    print(f"Error al conectar con la API. Código: {respuesta.status_code}")


TABLA DE PREDICCIONES XGBOOST



,Fecha,Local,Visitante,% Prob. Ambos Anotan,Predicción
0,2026-03-20,Cagliari,Napoli,32.59%,NO
1,2026-03-20,Genoa,Udinese,60.42%,SÍ
2,2026-03-21,Parma,Cremonese,69.12%,SÍ
3,2026-03-21,Milan,Torino,70.12%,SÍ
4,2026-03-21,Juventus,Sassuolo,49.65%,NO
5,2026-03-22,Como,Pisa,64.29%,SÍ
6,2026-03-22,Atalanta,Verona,41.87%,NO
7,2026-03-22,Bologna,Lazio,54.34%,SÍ
8,2026-03-22,Roma,Lecce,24.82%,NO
9,2026-03-22,Fiorentina,Inter,23.43%,NO
